In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
from pathlib import Path
from torch.utils.data import WeightedRandomSampler
from fastai.callback.tracker import SaveModelCallback, Recorder
import matplotlib.pyplot as plt
import numpy as np
import cv2
import random
import torch
from mtrain.neg_mask.model.datasets.blur_pad_dl import random_tfm, BlurPadDataset
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import cv2
import random
from mtrain.utils import show, mkdir, DiskImage, DiskBooleanMask
from pytorch_grad_cam import (
    GradCAM,
)
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
from tqdm import tqdm
from mtrain.neg_mask.model.show import (
    get_preds_for_ds,
    show_classification_report,
    show_confusion_matrix,
    show_confusion_matrix_using_preds,
)
from mtrain.neg_mask.model.datasets.blur_pad_dl import CropTfmsOutsideBbox
from functools import partial
from sklearn.model_selection import train_test_split
from fastai.basics import DataLoaders, default_device
from mtrain.denorm import denormalize_imagenet, denormalize_4chan_imagenet
from mtrain.utils import show, it_chain
from fastai.callback.all import ProgressCallback
from fastai.basics import F1Score, Precision, Recall, CrossEntropyLossFlat
from fastai.vision.all import vision_learner, xresnet18
from mtrain.utils import globL, mkdir
import json

In [ ]:
FOVEATED_DS_PATH = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/training/dataset"
)
DS_PATH = FOVEATED_DS_PATH

In [ ]:
def get_learner(dls):
    learn = vision_learner(
        dls,
        xresnet18,
        metrics=[F1Score(average="macro"), Precision(), Recall()],
        loss_func=CrossEntropyLossFlat(CLS_WEIGHT),
        n_out=2,
        normalize=False,
        n_in=3,
        pretrained=True,
    )
    learn.remove_cb(Recorder)
    learn.remove_cb(SaveModelCallback)
    learn.add_cb(Recorder())
    learn.add_cbs([SaveModelCallback(monitor="f1_score", fname="best")])
    learn = learn.remove_cb(ProgressCallback)
    return learn


def get_denormalized(tens):
    image, mask = None, None
    image = denormalize_imagenet(tens)
    image = image.permute([1, 2, 0]).numpy()
    if mask is not None:
        mask = mask.numpy()
    return image, mask


def show_gradcam_for_image(
    learn, input_tensor, target_label_idx=None, layer_name="0.7.1.conv1"
):
    target_layers = [learn.model.get_submodule(layer_name)]
    img_arr, _ = get_denormalized(input_tensor[0])

    targets = [ClassifierOutputTarget(target_label_idx)]

    with GradCAM(model=learn.model, target_layers=target_layers) as cam:
        grayscale_cam = cam(input_tensor=input_tensor, targets=targets)
        grayscale_cam = grayscale_cam[0, :]
        print(img_arr.shape, grayscale_cam.shape)
        visualization = show_cam_on_image(img_arr, grayscale_cam, use_rgb=True)
        model_outputs = cam.outputs

        return visualization, img_arr, model_outputs


def show_reports(learner):
    preds = learner.get_preds(dl=learner.dls.valid, with_decoded=True, with_loss=True)
    probs, targs, decoded, losses = preds
    labels = list(BlurPadDataset.LABEL_BY_IDX.keys())
    show_classification_report(probs, targs, labels)
    show_confusion_matrix_using_preds(probs, targs, labels)
    return probs, targs, decoded, losses


In [ ]:
CLS_WEIGHT = torch.tensor([1.0, 1.0]).float().to("mps")


def get_weight(p, taco_weight=1.0, manual_weight=1.2, default_weight=1.0):
    p = Path(p)
    if "taco" in p.name:
        return taco_weight
    if "manual" in p.name:
        return manual_weight
    else:
        return default_weight


def get_dls(
    num_samples,
    tfm,
    crop_size=224,
    ds_path=DS_PATH,
    min_area=35,
    min_bbox_length=3,
    max_area=None,
    path_filter=None,
):

    image_paths = list((ds_path / "train").glob("*.jpg"))[:num_samples]
    if path_filter is not None:
        image_paths = list(filter(path_filter, image_paths))
    stratify = [BlurPadDataset.label_func(p) for p in image_paths]
    train_paths, valid_paths = train_test_split(
        image_paths, test_size=0.2, stratify=stratify, random_state=42
    )

    train_ds = BlurPadDataset(
        train_paths,
        ds_path / "masks",
        crop_size,
        False,
        crop_mutator=tfm,
        bbox_pad=3,
        min_area=min_area,
        min_bbox_length=min_bbox_length,
        max_area=max_area,
    )
    valid_ds = BlurPadDataset(
        valid_paths,
        ds_path / "masks",
        crop_size,
        True,
        crop_mutator=tfm,
        bbox_pad=3,
        min_area=min_area,
        min_bbox_length=min_bbox_length,
        max_area=max_area,
    )
    image_paths = train_ds.image_paths
    train_weights = [get_weight(p) for p in image_paths]
    train_sampler = WeightedRandomSampler(
        weights=train_weights, num_samples=len(image_paths), replacement=True
    )
    dls = DataLoaders.from_dsets(
        train_ds,
        valid_ds,
        device=default_device(),
        num_workers=4,
        bs=16,
        # pin_memory=True,
        persistent_workers=True,
        dl_kwargs=[
            {"sampler": train_sampler, "shuffle": False},
            {"shuffle": False},  # Validation defaults
        ],
    )  # don't respawn workers each epoch)
    return dls


def vis_sample_ds(ds, idx):
    tens, targ = ds[idx]
    print("target", targ)
    print("shape", tens.shape)
    img, _ = get_denormalized(tens)
    plt.imshow(img, cmap="gray")
    plt.show()


def vis_sample(dls, idx):
    vis_sample_ds(dls.train_ds, idx)

In [ ]:
# to counter the problem of the model focusing on texture/noise
# we decrease the probability of adding noise with each sweep while maintaining accuracy
# the next step is to remove overwrite noise
# then next is decreasing the add noise frequency
# first i would need to seee the performance of the model
#  on different types of aux transforms (step down? gaussian? blur?)
# our final model has no noise, and one kind of step down function
# we need to test it on all transforms and find the winner
# for each we do successive training by decreasing the add_noise chance parameter
def blur_tfm(
    cropped_image, mask, inner_bbox, add_noise_chance, blur_kernel_sz, blur_sigma
):
    add_noise = random.random() < add_noise_chance
    tfm = CropTfmsOutsideBbox(cropped_image, inner_bbox)
    tfm = tfm.overwrite_with_blur(blur_kernel_sz, blur_sigma)
    if add_noise:
        tfm = tfm.add_noise(20)
    return tfm.crop


def step_down_tfm(cropped_image, mask, inner_bbox, add_noise_chance, ratio):
    add_noise = random.random() < add_noise_chance
    tfm = CropTfmsOutsideBbox(cropped_image, inner_bbox)
    tfm = tfm.step_down(ratio)
    if add_noise:
        tfm = tfm.add_noise(20)
    return tfm.crop, mask


def step_down_gauss_tfm(cropped_image, mask, inner_bbox, add_noise_chance, min_value):
    add_noise = random.random() < add_noise_chance
    tfm = CropTfmsOutsideBbox(cropped_image, inner_bbox)
    tfm = tfm.step_down_gaussian(min_value)
    if add_noise:
        tfm = tfm.add_noise(20)
    return tfm.crop

In [ ]:
def get_initialised_learner():
    dls = get_dls(100, random_tfm)
    learner = get_learner(dls)
    MODELS_DIR = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/models")
    path = MODELS_DIR / "foveated-224" / "iter-7-xresnet18.pth"
    state_dict = torch.load(path)
    learner.model.load_state_dict(state_dict)
    return learner

In [ ]:
LAST_LAYER_NAME = "0.7.1.convpath.1.0"
learner = get_initialised_learner()
_ = learner.model.to("mps")

In [ ]:
def get_all_as_valid_ds(
    num_samples,
    tfm,
    crop_size=224,
    ds_path=DS_PATH,
    min_area=35,
    min_bbox_length=3,
    max_area=None,
    path_filter=None,
):

    image_paths = list((ds_path / "train").glob("*.jpg"))[:num_samples]
    if path_filter is not None:
        image_paths = list(filter(path_filter, image_paths))

    return BlurPadDataset(
        image_paths,
        ds_path / "masks",
        crop_size,
        True,
        crop_mutator=tfm,
        bbox_pad=3,
        min_area=min_area,
        min_bbox_length=min_bbox_length,
        max_area=max_area,
    )

In [ ]:
st_ed_tfm = partial(step_down_tfm, ratio=0.5)
st_ed_tfm0 = partial(st_ed_tfm, add_noise_chance=-1)


def rm_mapi_walls(path):
    if "mapillary" in path.name:
        return False
    else:
        return True
valid_ds = get_all_as_valid_ds(200000, st_ed_tfm0, path_filter=rm_mapi_walls)

In [ ]:

SAMPLES_BASE = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/training"
)
MANUAL_L1_PATH = SAMPLES_BASE / "manually_annotated" / "L1"
TACO_L1_PATH = SAMPLES_BASE / "taco" / "L1"
WALLS_MAPILLARY_L1_PATH = SAMPLES_BASE / "walls-mapillary" / "L1"


def get_original_path(ds_path: Path | str):
    ds_path = Path(ds_path)
    without_label = "_".join((ds_path.stem).split("_")[1:])
    if without_label.endswith("_manual"):
        name = without_label[:-7]
        path = MANUAL_L1_PATH / name
    elif without_label.endswith("_taco"):
        name = without_label[:-5]
        path = TACO_L1_PATH / name
    elif without_label.endswith("_mapillary-walls"):
        name = without_label[: -len("_mapillary-walls")]
        path = WALLS_MAPILLARY_L1_PATH / name
    else:
        raise Exception(f"no useful suffix in {without_label} path={ds_path}")
    return path


def show_originals(ds_path: Path | str):
    ds_path = Path(ds_path)
    orig_path = get_original_path(ds_path)
    print("original path", orig_path)
    if not orig_path.exists():
        raise Exception(f"original path {orig_path} does not exist")
    image = DiskImage.load(orig_path / "image.jpg")
    mask = DiskBooleanMask.load(orig_path / "mask.png")
    with open(orig_path / "model.json") as f:
        mres = json.load(f)
        area = mres["bbox"]["h"] * mres["bbox"]["w"]
        print(mres["bbox"], area)
    print("ds_path", ds_path.stem)
    show([image, mask], (20, 20))

In [ ]:
class EmbeddingCollector:
    def __init__(self, model, layer_idx):
        # We hook into layer (1): the Flatten layer
        # This gives us the 1024-dim vector (Concat of Avg & Max pool)
        self.target_layer = model[1][layer_idx] 
        self.hook = self.target_layer.register_forward_hook(self.hook_fn)
        self.stored = None

    def hook_fn(self, m, i, o):
        # o is the output of the Flatten layer
        self.stored = o.detach().cpu()

    def remove(self): self.hook.remove()

def precompute_all(learn, valid_ds, layer_idx=1):
    collector = EmbeddingCollector(learn.model, layer_idx)
    embs = []
    
    learn.model.eval()
    test_dl = learn.dls.test_dl(valid_ds, with_labels=True, shuffle=False)
    with torch.no_grad():
        for b in tqdm(test_dl):
            _ = learn.model(b[0])
            embs.append(collector.stored)
    
    collector.remove()
    return torch.cat(embs)

# Run it
# embs, file_list = precompute_all(learn, dls.valid)
# torch.save({'embs': embs, 'files': file_list}, 'validation_vectors.pt') 

In [ ]:
learner.eval()
embs_after_linear = precompute_all(learner, valid_ds, 4)
embs_after_relu = precompute_all(learner, valid_ds, 5)

In [ ]:
torch.save({"embs": embs_after_linear, "paths": valid_ds.image_paths}, "./embs_after_linear.pt")
torch.save({"embs": embs_after_relu, "paths": valid_ds.image_paths}, "./embs_after_relu.pt")

In [ ]:
import torch.nn.functional as F

def query_similar_embs(query_idx, embs, file_list, top_k=10):
    """
    Pass the index of an image you KNOW is a false positive 
    due to straight lines (e.g., a fence).
    """
    query_vec = embs[query_idx].unsqueeze(0)
    
    # Cosine similarity against the whole validation set
    sims = F.cosine_similarity(query_vec, embs)
    
    # Get top matches
    vals, idxs = torch.topk(sims, top_k)
    # return vals, idxs
    
    return idxs
    # return [(file_list[i], sims[i].item()) for i in idxs]

def get_images_to_show(ds, preds, targs, idxes):
    # tens, targ = ds[idx]
    # print("target", targ)
    # print("shape", tens.shape)
    # img, _ = get_denormalized(tens)
    # plt.imshow(img, cmap="gray")
    pred_trash_targ_other, pred_other_targ_trash = [], []
    correct = []
    for i in idxes:
        tens, _ = ds[i]
        img, _ = get_denormalized(tens)
        title = f"{i}/{targs[i]}"

        if preds[i] == 0 and targs[i] == 1:
            pred_other_targ_trash.append((img, title))
        elif preds[i] == 1 and targs[i] == 0:
            pred_trash_targ_other.append((img, title))
        else:
            # its the right answer:
            correct.append((img, title))
    return correct, pred_other_targ_trash, pred_trash_targ_other

# Example: If index 42 is a fence that was called 'trash'
# matches = find_edge_patterns(42, embs, file_list)

In [ ]:
test_dl = learner.dls.test_dl(valid_ds, with_labels=True, shuffle=False)
preds = learner.get_preds(dl=test_dl, with_decoded=True, with_loss=True)
probs, targs, decoded, losses = preds
sorted_losses = list(reversed(sorted([(loss, i) for i, loss in enumerate(losses)])))
top_loss_idxs = [sl[1] for sl in sorted_losses]
min_losses = [sl[1] for sl in reversed(sorted_losses)]

In [ ]:
idx = top_loss_idxs[0]
print("index", idx)
vis_sample_ds(valid_ds, idx)

In [ ]:
valid_ds.image_paths[idx]

In [ ]:
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

def cluster_embeddings(embs, n_clusters=10):
    # 1. Convert to numpy and normalize
    # Normalization ensures clusters are based on 'pattern' not 'intensity'
    X = F.normalize(embs, p=2, dim=1).numpy()
    
    # 2. PCA to reduce noise (1024 -> 50)
    pca = PCA(n_components=50, random_state=42)
    X_pca = pca.fit_transform(X)
    
    # 3. K-Means
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(X_pca)
    
    return cluster_labels, X_pca

def get_cluster_indices(embs, n_clusters=20, n_pca=200):
    # 1. Normalize and Reduce (using your 200-dim finding)
    X = F.normalize(embs, p=2, dim=1).numpy()
    pca = PCA(n_components=n_pca, random_state=42)
    X_pca = pca.fit_transform(X)
    
    # 2. Run K-Means
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_pca)
    
    # 3. Create List of Lists (Indices per cluster)
    # Each sub-list contains the original indices of the images in that cluster
    cluster_list = [np.where(labels == i)[0].tolist() for i in range(n_clusters)]
    
    return cluster_list

def plot_clusters(X_reduced, cluster_labels):
    # Reduce to 2D for plotting
    tsne = TSNE(n_components=2, random_state=42)
    X_2d = tsne.fit_transform(X_reduced)
    
    plt.figure(figsize=(12, 8))
    scatter = plt.scatter(X_2d[:, 0], X_2d[:, 1], c=cluster_labels, cmap='tab20', s=5)
    plt.legend(*scatter.legend_elements(), title="Clusters")
    plt.title("Embedding Clusters (t-SNE)")
    plt.show()

def accuracy_of_cluster(idxes):
    correct = (decoded[idxes] == targs[idxes]).sum()
    return correct / len(idxes)
# plot_clusters(x_reduced, cluster_ids)
# Usage
# cluster_ids, x_reduced = cluster_embeddings(embs, n_clusters=15)

In [ ]:
after_relu_ids = get_cluster_indices(embs_after_relu, n_clusters=50)
after_linear_ids = get_cluster_indices(embs_after_linear, 50)

In [ ]:
after_relu_accs = [(accuracy_of_cluster(cids),i) for i, cids in enumerate(after_relu_ids)]
after_linear_accs = [(accuracy_of_cluster(cids),i) for i, cids in enumerate(after_linear_ids)]

In [ ]:
sorted(after_relu_accs)[:10]

In [ ]:
sorted(after_linear_accs)[:10]

for this particular ReLU block, the accuracy seems to have improved for everyone other than the first group.  
The accuracy is more averaged out? Although the effect is minor.  

In [ ]:
def split_to_images_titles(images):
    return [i[0] for i in images], [i[1] for i in images]

In [ ]:
correct, pred_other_targ_trash, pred_trash_targ_other = get_images_to_show(valid_ds, decoded, targs, after_linear_ids[13][:20])

In [ ]:
images, titles = split_to_images_titles(pred_other_targ_trash)
show(images, (20,20), 4, titles=titles)

In [ ]:
images, titles = split_to_images_titles(pred_trash_targ_other)
show(images, (6,4), titles=titles)

In [ ]:
images, titles = split_to_images_titles(correct)
show(images, (20,20), ncols=4, titles=titles)

These are fair I would say, not too bad. It seems like we have small white objects in this group in focus. backgrounds are varying but are predominently gray. Roadlike.  
Let's try to find them in the linear cluster ids. 
I'll check out the image 1670.  

In [ ]:
vis_sample_ds(valid_ds, 1670)

In [ ]:
for i, lst in enumerate(after_relu_ids):
    if 1670 in lst:
        print("found at index", i)
        break
print(after_relu_ids[i])
print(after_linear_ids[13])

In [ ]:
for i, ac in enumerate(after_relu_accs):
    if ac[1] == 35:
        print("found accuracy at", i)
        break

# accuracy of the group we are in (quite good.)
print(after_relu_accs[i])

In [ ]:
correct, pred_other_targ_trash, pred_trash_targ_other = get_images_to_show(valid_ds, decoded, targs, after_relu_ids[35])

In [ ]:
images, titles = split_to_images_titles(pred_other_targ_trash)
# no wrong pred was other in this
show(images, (20,20), 4, titles=titles)

In [ ]:
# here the model is not looking at the right place
# for some of them its actually looking at the correct place (the first image). 
# for this specific one, its quite wrong.  
viz, img, mo = show_gradcam_for_image(learner, valid_ds[4718][0].unsqueeze(0), 1, "0.7.1.convpath.1.0")
show([viz, img])

In [ ]:
images, titles = split_to_images_titles(pred_trash_targ_other)
# print(len(images))
show(images, (20,20), 4, titles=titles)

In [ ]:
images, titles = split_to_images_titles(correct[80:100])
# print(len(images))
show(images, (20,20), ncols=4, titles=titles)

We see that the ReLU layer has ALREADY grouped these objects in a TRASH only bucket.  
The only false positive here is a rock. I'm not sure why it is present in this group. What are the characteristics of this group?  

To me, these look like good resolution objects, in the same group. Interesting.  

What can I check next? I can check where our object lands after the next layer. Or I can check where the other objects in the initial group land. Checking the next layer requires more embedding calculations.  
So the interesting part here is that I can see which group an object lands in after every layer. Or I can see how the groups react. Seeing where the object lands in the new group is easier I would say. Let's continue that (we ll do group level stuff later.)